In [1]:
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import ast

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
df = pd.read_csv('../data/appartments.csv').drop(22)

In [4]:
df.head()

,PropertyName,PropertySubName,NearbyLocations,LocationAdvantages,Link,PriceDetails,TopFacilities
0,Smartworld One DXP,"2, 3, 4 BHK Apartment in Sector 113, Gurgaon","['Bajghera Road', 'Palam Vihar Halt', 'DPSG Pa...","{'Bajghera Road': '800 Meter', 'Palam Vihar Ha...",https://www.99acres.com/smartworld-one-dxp-sec...,"{'2 BHK': {'building_type': 'Apartment', 'area...","['Swimming Pool', 'Salon', 'Restaurant', 'Spa'..."
1,M3M Crown,"3, 4 BHK Apartment in Sector 111, Gurgaon","['DPSG Palam Vihar Gurugram', 'The NorthCap Un...","{'DPSG Palam Vihar Gurugram': '1.4 Km', 'The N...",https://www.99acres.com/m3m-crown-sector-111-g...,"{'3 BHK': {'building_type': 'Apartment', 'area...","['Bowling Alley', 'Mini Theatre', 'Manicured G..."
2,Adani Brahma Samsara Vilasa,"Land, 3, 4 BHK Independent Floor in Sector 63,...","['AIPL Business Club Sector 62', 'Heritage Xpe...","{'AIPL Business Club Sector 62': '2.7 Km', 'He...",https://www.99acres.com/adani-brahma-samsara-v...,{'3 BHK': {'building_type': 'Independent Floor...,"['Terrace Garden', 'Gazebo', 'Fountain', 'Amph..."
3,Sobha City,"2, 3, 4 BHK Apartment in Sector 108, Gurgaon","['The Shikshiyan School', 'WTC Plaza', 'Luxus ...","{'The Shikshiyan School': '2.9 KM', 'WTC Plaza...",https://www.99acres.com/sobha-city-sector-108-...,"{'2 BHK': {'building_type': 'Apartment', 'area...","['Swimming Pool', 'Volley Ball Court', 'Aerobi..."
4,Signature Global City 93,"2, 3 BHK Independent Floor in Sector 93 Gurgaon","['Pranavananda Int. School', 'DLF Site central...","{'Pranavananda Int. School': '450 m', 'DLF Sit...",https://www.99acres.com/signature-global-city-...,{'2 BHK': {'building_type': 'Independent Floor...,"['Mini Theatre', 'Doctor on Call', 'Concierge ..."


In [5]:
df.iloc[2].NearbyLocations

"['AIPL Business Club Sector 62', 'Heritage Xperiential Learning School', 'CK Birla Hospital', 'Paras Trinity Mall Sector 63', 'Rapid Metro Station Sector 56']"

In [6]:
df.iloc[2].LocationAdvantages

"{'AIPL Business Club Sector 62': '2.7 Km', 'Heritage Xperiential Learning School': '2 Km', 'CK Birla Hospital': '2.5 Km', 'Paras Trinity Mall Sector 63': '3.5 Km', 'Rapid Metro Station Sector 56': '3.8 Km', 'De Adventure Park': '6.8 Km', 'Golf Course Ext Rd': '99 Meter', 'DoubleTree by Hilton Hotel Gurgaon': '3.6 Km', 'KIIT College of Engineering Sohna Road': '8.4 Km', 'Mehrauli-Gurgaon Road': '11.8 Km', 'Indira Gandhi International Airport': '21.1 Km', 'Nirvana Rd': '160 Meter', 'TERI Golf Course': '8.7 Km'}"

In [7]:
# 1st recommender system
df[['PropertyName','TopFacilities']]['TopFacilities'][0]

"['Swimming Pool', 'Salon', 'Restaurant', 'Spa', 'Cafeteria', 'Sun Deck', '24x7 Security', 'Club House', 'Gated Community']"

In [8]:
def extract_list(s):
    return re.findall(r"'(.*?)'", s)

df['TopFacilities'] = df['TopFacilities'].apply(extract_list)

In [9]:
df['TopFacilities'][0]

['Swimming Pool',
 'Salon',
 'Restaurant',
 'Spa',
 'Cafeteria',
 'Sun Deck',
 '24x7 Security',
 'Club House',
 'Gated Community']

In [10]:
df['FacilitiesStr'] = df['TopFacilities'].apply(' '.join)

In [11]:
df.head()

,PropertyName,PropertySubName,NearbyLocations,LocationAdvantages,Link,PriceDetails,TopFacilities,FacilitiesStr
0,Smartworld One DXP,"2, 3, 4 BHK Apartment in Sector 113, Gurgaon","['Bajghera Road', 'Palam Vihar Halt', 'DPSG Pa...","{'Bajghera Road': '800 Meter', 'Palam Vihar Ha...",https://www.99acres.com/smartworld-one-dxp-sec...,"{'2 BHK': {'building_type': 'Apartment', 'area...","[Swimming Pool, Salon, Restaurant, Spa, Cafete...",Swimming Pool Salon Restaurant Spa Cafeteria S...
1,M3M Crown,"3, 4 BHK Apartment in Sector 111, Gurgaon","['DPSG Palam Vihar Gurugram', 'The NorthCap Un...","{'DPSG Palam Vihar Gurugram': '1.4 Km', 'The N...",https://www.99acres.com/m3m-crown-sector-111-g...,"{'3 BHK': {'building_type': 'Apartment', 'area...","[Bowling Alley, Mini Theatre, Manicured Garden...",Bowling Alley Mini Theatre Manicured Garden Sw...
2,Adani Brahma Samsara Vilasa,"Land, 3, 4 BHK Independent Floor in Sector 63,...","['AIPL Business Club Sector 62', 'Heritage Xpe...","{'AIPL Business Club Sector 62': '2.7 Km', 'He...",https://www.99acres.com/adani-brahma-samsara-v...,{'3 BHK': {'building_type': 'Independent Floor...,"[Terrace Garden, Gazebo, Fountain, Amphitheatr...",Terrace Garden Gazebo Fountain Amphitheatre Pa...
3,Sobha City,"2, 3, 4 BHK Apartment in Sector 108, Gurgaon","['The Shikshiyan School', 'WTC Plaza', 'Luxus ...","{'The Shikshiyan School': '2.9 KM', 'WTC Plaza...",https://www.99acres.com/sobha-city-sector-108-...,"{'2 BHK': {'building_type': 'Apartment', 'area...","[Swimming Pool, Volley Ball Court, Aerobics Ce...",Swimming Pool Volley Ball Court Aerobics Centr...
4,Signature Global City 93,"2, 3 BHK Independent Floor in Sector 93 Gurgaon","['Pranavananda Int. School', 'DLF Site central...","{'Pranavananda Int. School': '450 m', 'DLF Sit...",https://www.99acres.com/signature-global-city-...,{'2 BHK': {'building_type': 'Independent Floor...,"[Mini Theatre, Doctor on Call, Concierge Servi...",Mini Theatre Doctor on Call Concierge Service ...


In [12]:
df['FacilitiesStr'][0]

'Swimming Pool Salon Restaurant Spa Cafeteria Sun Deck 24x7 Security Club House Gated Community'

In [13]:
# tfidf vectorization
tfidf_vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))

In [14]:
tfidf_matrix = tfidf_vectorizer.fit_transform(df['FacilitiesStr'])

In [15]:
tfidf_matrix.toarray()[0]

array([0.        , 0.        , 0.        , 0.18809342, 0.18809342,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.     

In [16]:
cosine_sim1 = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [17]:
cosine_sim1.shape

(246, 246)

In [18]:
df[['PropertyName','PriceDetails']]['PriceDetails'][1]

"{'3 BHK': {'building_type': 'Apartment', 'area_type': 'Super Built-up Area', 'area': '1,605 - 2,170 sq.ft.', 'price-range': '₹ 2.2 - 3.03 Cr'}, '4 BHK': {'building_type': 'Apartment', 'area_type': 'Super Built-up Area', 'area': '2,248 - 2,670 sq.ft.', 'price-range': '₹ 3.08 - 3.73 Cr'}}"

In [19]:
def recommend_properties(property_name, cosine_sim=cosine_sim1):
    # Get the index of the property that matches the name
    idx = df.index[df['PropertyName'] == property_name].tolist()[0]

    # Get the pairwise similarity scores with that property
    sim_scores = list(enumerate(cosine_sim1[idx]))

    # Sort the properties based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the 10 most similar properties
    sim_scores = sim_scores[1:6]

    # Get the property indices
    property_indices = [i[0] for i in sim_scores]
    
    recommendations_df = pd.DataFrame({
        'PropertyName': df['PropertyName'].iloc[property_indices],
        'SimilarityScore': sim_scores
    })

    # Return the top 10 most similar properties
    return recommendations_df

In [20]:
recommend_properties("DLF The Arbour")

,PropertyName,SimilarityScore
64,Ace Palm Floors,"(63, 0.4529382062441955)"
217,Yashika 104,"(216, 0.4199606322926784)"
93,JMS The Nation,"(92, 0.4166584649363288)"
154,India Rashtra,"(153, 0.398954234680194)"
0,Smartworld One DXP,"(0, 0.38885046199432893)"


In [21]:
# 2nd recommender system
import pandas as pd
import json

# Load the dataset
df_appartments = pd.read_csv('../data/appartments.csv').drop(22)

# Function to parse and extract the required features from the PriceDetails column
def refined_parse_modified_v2(detail_str):
    try:
        details = json.loads(detail_str.replace("'", "\""))
    except:
        return {}

    extracted = {}
    for bhk, detail in details.items():
        # Extract building type
        extracted[f'building type_{bhk}'] = detail.get('building_type')

        # Parsing area details
        area = detail.get('area', '')
        area_parts = area.split('-')
        if len(area_parts) == 1:
            try:
                value = float(area_parts[0].replace(',', '').replace(' sq.ft.', '').strip())
                extracted[f'area low {bhk}'] = value
                extracted[f'area high {bhk}'] = value
            except:
                extracted[f'area low {bhk}'] = None
                extracted[f'area high {bhk}'] = None
        elif len(area_parts) == 2:
            try:
                extracted[f'area low {bhk}'] = float(area_parts[0].replace(',', '').replace(' sq.ft.', '').strip())
                extracted[f'area high {bhk}'] = float(area_parts[1].replace(',', '').replace(' sq.ft.', '').strip())
            except:
                extracted[f'area low {bhk}'] = None
                extracted[f'area high {bhk}'] = None

        # Parsing price details
        price_range = detail.get('price-range', '')
        price_parts = price_range.split('-')
        if len(price_parts) == 2:
            try:
                extracted[f'price low {bhk}'] = float(price_parts[0].replace('₹', '').replace(' Cr', '').replace(' L', '').strip())
                extracted[f'price high {bhk}'] = float(price_parts[1].replace('₹', '').replace(' Cr', '').replace(' L', '').strip())
                if 'L' in price_parts[0]:
                    extracted[f'price low {bhk}'] /= 100
                if 'L' in price_parts[1]:
                    extracted[f'price high {bhk}'] /= 100
            except:
                extracted[f'price low {bhk}'] = None
                extracted[f'price high {bhk}'] = None

    return extracted
# Apply the refined parsing and generate the new DataFrame structure
data_refined = []

for _, row in df_appartments.iterrows():
    features = refined_parse_modified_v2(row['PriceDetails'])
    
    # Construct a new row for the transformed dataframe
    new_row = {'PropertyName': row['PropertyName']}
    
    # Populate the new row with extracted features
    for config in ['1 BHK', '2 BHK', '3 BHK', '4 BHK', '5 BHK', '6 BHK', '1 RK', 'Land']:
        new_row[f'building type_{config}'] = features.get(f'building type_{config}')
        new_row[f'area low {config}'] = features.get(f'area low {config}')
        new_row[f'area high {config}'] = features.get(f'area high {config}')
        new_row[f'price low {config}'] = features.get(f'price low {config}')
        new_row[f'price high {config}'] = features.get(f'price high {config}')
    
    data_refined.append(new_row)

df_final_refined_v2 = pd.DataFrame(data_refined).set_index('PropertyName')

In [22]:
df_final_refined_v2['building type_Land'] = df_final_refined_v2['building type_Land'].replace({'':'Land'})

In [23]:
df['PriceDetails'][10]

"{'2 BHK': {'building_type': 'Independent Floor', 'area_type': 'Carpet Area', 'area': '1,055 sq.ft.', 'price-range': '₹ 1.05 - 1.5 Cr'}, '3 BHK': {'building_type': 'Independent Floor', 'area_type': 'Carpet Area', 'area': '1,325 - 1,525 sq.ft.', 'price-range': '₹ 1.35 - 1.84 Cr'}}"

In [24]:
df_final_refined_v2

,building type_1 BHK,area low 1 BHK,area high 1 BHK,price low 1 BHK,price high 1 BHK,building type_2 BHK,area low 2 BHK,area high 2 BHK,price low 2 BHK,price high 2 BHK,building type_3 BHK,area low 3 BHK,area high 3 BHK,price low 3 BHK,price high 3 BHK,building type_4 BHK,area low 4 BHK,area high 4 BHK,price low 4 BHK,price high 4 BHK,building type_5 BHK,area low 5 BHK,area high 5 BHK,price low 5 BHK,price high 5 BHK,building type_6 BHK,area low 6 BHK,area high 6 BHK,price low 6 BHK,price high 6 BHK,building type_1 RK,area low 1 RK,area high 1 RK,price low 1 RK,price high 1 RK,building type_Land,area low Land,area high Land,price low Land,price high Land
PropertyName,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,None,NaN,NaN,NaN,NaN,Apartment,1370.0,1370.0,2.0000,2.40,Apartment,1850.0,2050.0,2.25,3.59,Apartment,2600.0,2600.0,3.24,4.56,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
M3M Crown,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,Apartment,1605.0,2170.0,2.20,3.03,Apartment,2248.0,2670.0,3.08,3.73,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
Adani Brahma Samsara Vilasa,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,Independent Floor,1800.0,3150.0,2.43,15.75,Independent Floor,2750.0,4500.0,3.36,22.50,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,Land,500.0,4329.0,2.05,41.13
Sobha City,None,NaN,NaN,NaN,NaN,Apartment,1381.0,1692.0,1.5500,3.21,Apartment,1711.0,2343.0,1.76,4.79,Apartment,2423.0,2963.0,2.50,6.06,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
Signature Global City 93,None,NaN,NaN,NaN,NaN,Independent Floor,981.0,1118.0,0.9301,1.06,Independent Floor,1235.0,1530.0,1.12,1.45,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DLF Princeton Estate,None,NaN,NaN,NaN,NaN,Apartment,964.0,964.0,NaN,NaN,Apartment,1127.0,1127.0,NaN,NaN,Apartment,1562.0,1562.0,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
Pyramid Urban Homes 2,Apartment,335.0,398.0,23.45,0.2786,Apartment,500.0,625.0,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
Satya The Hermitage,None,NaN,NaN,NaN,NaN,Apartment,1450.0,1450.0,NaN,NaN,Apartment,1991.0,1991.0,NaN,NaN,Apartment,2639.0,4711.0,1.20,2.14,Apartment,4731.0,4731.0,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN


In [25]:
categorical_columns = df_final_refined_v2.select_dtypes(include=['object']).columns.tolist()

In [26]:
categorical_columns

['building type_1 BHK',
 'building type_2 BHK',
 'building type_3 BHK',
 'building type_4 BHK',
 'building type_5 BHK',
 'building type_6 BHK',
 'building type_1 RK',
 'building type_Land']

In [27]:
ohe_df = pd.get_dummies(df_final_refined_v2, columns=categorical_columns, drop_first=True)

In [28]:
ohe_df.fillna(0,inplace=True)

In [29]:
ohe_df

,area low 1 BHK,area high 1 BHK,price low 1 BHK,price high 1 BHK,area low 2 BHK,area high 2 BHK,price low 2 BHK,price high 2 BHK,area low 3 BHK,area high 3 BHK,price low 3 BHK,price high 3 BHK,area low 4 BHK,area high 4 BHK,price low 4 BHK,price high 4 BHK,area low 5 BHK,area high 5 BHK,price low 5 BHK,price high 5 BHK,area low 6 BHK,area high 6 BHK,price low 6 BHK,price high 6 BHK,area low 1 RK,area high 1 RK,price low 1 RK,price high 1 RK,area low Land,area high Land,price low Land,price high Land,building type_1 BHK_Service Apartment,building type_2 BHK_Independent Floor,building type_2 BHK_Service Apartment,building type_3 BHK_Independent Floor,building type_3 BHK_Service Apartment,building type_3 BHK_Villa,building type_4 BHK_Independent Floor,building type_4 BHK_Villa,building type_5 BHK_Independent Floor,building type_5 BHK_Villa,building type_6 BHK_Villa
PropertyName,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,0.0,0.0,0.00,0.0000,1370.0,1370.0,2.0000,2.40,1850.0,2050.0,2.25,3.59,2600.0,2600.0,3.24,4.56,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,False,False,False,False,False,False,False,False,False,False,False
M3M Crown,0.0,0.0,0.00,0.0000,0.0,0.0,0.0000,0.00,1605.0,2170.0,2.20,3.03,2248.0,2670.0,3.08,3.73,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,False,False,False,False,False,False,False,False,False,False,False
Adani Brahma Samsara Vilasa,0.0,0.0,0.00,0.0000,0.0,0.0,0.0000,0.00,1800.0,3150.0,2.43,15.75,2750.0,4500.0,3.36,22.50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,500.0,4329.0,2.05,41.13,False,False,False,True,False,False,True,False,False,False,False
Sobha City,0.0,0.0,0.00,0.0000,1381.0,1692.0,1.5500,3.21,1711.0,2343.0,1.76,4.79,2423.0,2963.0,2.50,6.06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,False,False,False,False,False,False,False,False,False,False,False
Signature Global City 93,0.0,0.0,0.00,0.0000,981.0,1118.0,0.9301,1.06,1235.0,1530.0,1.12,1.45,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,False,True,False,True,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DLF Princeton Estate,0.0,0.0,0.00,0.0000,964.0,964.0,0.0000,0.00,1127.0,1127.0,0.00,0.00,1562.0,1562.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,False,False,False,False,False,False,False,False,False,False,False
Pyramid Urban Homes 2,335.0,398.0,23.45,0.2786,500.0,625.0,0.0000,0.00,0.0,0.0,0.00,0.00,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,False,False,False,False,False,False,False,False,False,False,False
Satya The Hermitage,0.0,0.0,0.00,0.0000,1450.0,1450.0,0.0000,0.00,1991.0,1991.0,0.00,0.00,2639.0,4711.0,1.20,2.14,4731.0,4731.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,False,False,False,False,False,False,False,False,False,False,False


In [30]:
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()

# Apply the scaler to the entire dataframe
ohe_df_normalized = pd.DataFrame(scaler.fit_transform(ohe_df), columns=ohe_df.columns, index=ohe_df.index)

In [31]:
ohe_df_normalized.head()

,area low 1 BHK,area high 1 BHK,price low 1 BHK,price high 1 BHK,area low 2 BHK,area high 2 BHK,price low 2 BHK,price high 2 BHK,area low 3 BHK,area high 3 BHK,price low 3 BHK,price high 3 BHK,area low 4 BHK,area high 4 BHK,price low 4 BHK,price high 4 BHK,area low 5 BHK,area high 5 BHK,price low 5 BHK,price high 5 BHK,area low 6 BHK,area high 6 BHK,price low 6 BHK,price high 6 BHK,area low 1 RK,area high 1 RK,price low 1 RK,price high 1 RK,area low Land,area high Land,price low Land,price high Land,building type_1 BHK_Service Apartment,building type_2 BHK_Independent Floor,building type_2 BHK_Service Apartment,building type_3 BHK_Independent Floor,building type_3 BHK_Service Apartment,building type_3 BHK_Villa,building type_4 BHK_Independent Floor,building type_4 BHK_Villa,building type_5 BHK_Independent Floor,building type_5 BHK_Villa,building type_6 BHK_Villa
PropertyName,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,-0.252266,-0.169584,-0.105197,-0.082332,1.223499,1.020101,-0.173712,1.158423,0.553787,0.370864,0.807098,0.515061,0.602838,0.212073,0.383381,0.242019,-0.468954,-0.460463,-0.248049,-0.235915,-0.125582,-0.118934,-0.077649,-0.073387,-0.105157,-0.10253,-0.090521,-0.082725,-0.447044,-0.371421,-0.195703,-0.240489,-0.111111,-0.289310,-0.063888,-0.372678,-0.063888,-0.171139,-0.254824,-0.236208,-0.111111,-0.216353,-0.063888
M3M Crown,-0.252266,-0.169584,-0.105197,-0.082332,-0.893541,-0.896660,-0.283546,-0.387986,0.293086,0.472749,0.772095,0.342903,0.382746,0.243172,0.341443,0.108677,-0.468954,-0.460463,-0.248049,-0.235915,-0.125582,-0.118934,-0.077649,-0.073387,-0.105157,-0.10253,-0.090521,-0.082725,-0.447044,-0.371421,-0.195703,-0.240489,-0.111111,-0.289310,-0.063888,-0.372678,-0.063888,-0.171139,-0.254824,-0.236208,-0.111111,-0.216353,-0.063888
Adani Brahma Samsara Vilasa,-0.252266,-0.169584,-0.105197,-0.082332,-0.893541,-0.896660,-0.283546,-0.387986,0.500583,1.304803,0.933110,4.253341,0.696627,1.056178,0.414835,3.124152,-0.468954,-0.460463,-0.248049,-0.235915,-0.125582,-0.118934,-0.077649,-0.073387,-0.105157,-0.10253,-0.090521,-0.082725,0.369534,1.828502,0.643875,7.635886,-0.111111,-0.289310,-0.063888,2.683282,-0.063888,-0.171139,3.924283,-0.236208,-0.111111,-0.216353,-0.063888
Sobha City,-0.252266,-0.169584,-0.105197,-0.082332,1.240497,1.470610,-0.198425,1.680336,0.405879,0.619632,0.464065,0.883970,0.492166,0.373342,0.189416,0.483000,-0.468954,-0.460463,-0.248049,-0.235915,-0.125582,-0.118934,-0.077649,-0.073387,-0.105157,-0.10253,-0.090521,-0.082725,-0.447044,-0.371421,-0.195703,-0.240489,-0.111111,-0.289310,-0.063888,-0.372678,-0.063888,-0.171139,-0.254824,-0.236208,-0.111111,-0.216353,-0.063888
Signature Global City 93,-0.252266,-0.169584,-0.105197,-0.082332,0.622383,0.667529,-0.232468,0.295011,-0.100626,-0.070634,0.016022,-0.142827,-1.022842,-0.943016,-0.465873,-0.490563,-0.468954,-0.460463,-0.248049,-0.235915,-0.125582,-0.118934,-0.077649,-0.073387,-0.105157,-0.10253,-0.090521,-0.082725,-0.447044,-0.371421,-0.195703,-0.240489,-0.111111,3.456497,-0.063888,2.683282,-0.063888,-0.171139,-0.254824,-0.236208,-0.111111,-0.216353,-0.063888


In [32]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute the cosine similarity matrix
cosine_sim2 = cosine_similarity(ohe_df_normalized)

In [33]:
cosine_sim2.shape

(246, 246)

In [34]:
def recommend_properties_with_scores(property_name, top_n=247):
    
    # Get the similarity scores for the property using its name as the index
    sim_scores = list(enumerate(cosine_sim2[ohe_df_normalized.index.get_loc(property_name)]))
    
    # Sort properties based on the similarity scores
    sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get the indices and scores of the top_n most similar properties
    top_indices = [i[0] for i in sorted_scores[1:top_n+1]]
    top_scores = [i[1] for i in sorted_scores[1:top_n+1]]
    
    # Retrieve the names of the top properties using the indices
    top_properties = ohe_df_normalized.index[top_indices].tolist()
    
    # Create a dataframe with the results
    recommendations_df = pd.DataFrame({
        'PropertyName': top_properties,
        'SimilarityScore': top_scores
    })
    
    return recommendations_df

# Test the recommender function using a property name
recommend_properties_with_scores('M3M Golf Hills')

,PropertyName,SimilarityScore
0,AIPL The Peaceful Homes,0.955462
1,Smartworld One DXP,0.954670
2,Unitech Escape,0.953092
3,M3M Capital,0.951156
4,BPTP Terra,0.943128
...,...,...
240,Golden Park,-0.522391
241,Satya Merano Greens,-0.523660
242,ROF Normanton Park,-0.525129
243,BPTP Green Oaks,-0.525286


In [35]:
# 3rd recommender systems
df[['PropertyName','LocationAdvantages']]['LocationAdvantages'][0]

"{'Bajghera Road': '800 Meter', 'Palam Vihar Halt': '2.5 KM', 'DPSG Palam Vihar': '3.1 KM', 'Park Hospital': '3.1 KM', 'Gurgaon Railway Station': '4.9 KM', 'The NorthCap University': '5.4 KM', 'Dwarka Expy': '1.2 KM', 'Hyatt Place Gurgaon Udyog Vihar': '7.7 KM', 'Dwarka Sector 21, Metro Station': '7.2 KM', 'Pacific D21 Mall': '7.4 KM', 'Indira Gandhi International Airport': '14.7 KM', 'Hamoni Golf Camp': '6.2 KM', 'Fun N Food Waterpark': '8.8 KM', 'Accenture DDC5': '9 KM'}"

In [36]:
def distance_to_meters(distance_str):
    try:
        if 'Km' in distance_str or 'KM' in distance_str:
            return float(distance_str.split()[0]) * 1000
        elif 'Meter' in distance_str or 'meter' in distance_str:
            return float(distance_str.split()[0])
        else:
            return None
    except:
        return None

In [37]:
# Extract distances for each location
location_matrix = {}
for index, row in df.iterrows():
    distances = {}
    for location, distance in ast.literal_eval(row['LocationAdvantages']).items():
        distances[location] = distance_to_meters(distance)
    location_matrix[index] = distances

# Convert the dictionary to a dataframe
location_df = pd.DataFrame.from_dict(location_matrix, orient='index')

# Display the first few rows
location_df.head()

,Bajghera Road,Palam Vihar Halt,DPSG Palam Vihar,Park Hospital,Gurgaon Railway Station,The NorthCap University,Dwarka Expy,Hyatt Place Gurgaon Udyog Vihar,"Dwarka Sector 21, Metro Station",Pacific D21 Mall,Indira Gandhi International Airport,Hamoni Golf Camp,Fun N Food Waterpark,Accenture DDC5,DPSG Palam Vihar Gurugram,"Park Hospital, Palam Vihar",Palam Vihar Halt Railway Station,Dwarka Sector 21 Metro Station,Dwarka Expressway,Fun N Food Water Park,Tau DeviLal Sports Complex,Hyatt Place,Altrade Business Centre,AIPL Business Club Sector 62,Heritage Xperiential Learning School,CK Birla Hospital,Paras Trinity Mall Sector 63,Rapid Metro Station Sector 56,De Adventure Park,Golf Course Ext Rd,DoubleTree by Hilton Hotel Gurgaon,KIIT College of Engineering Sohna Road,Mehrauli-Gurgaon Road,Nirvana Rd,TERI Golf Course,The Shikshiyan School,WTC Plaza,Luxus Haritma Resort,BSF Golf Course,Rions Hospital,Gurgaon,Dwarka Sector 21,Nehru Stadium,Fun N Food WaterPark,IGI Airport,Vasant Kunj,Pranavananda Int. School,DLF Site central office,Holiday Inn Gurugram Sector 90,Krishna Hospital,Royal Institute Of Science,Sapphire 83 Mall,NH48,Garhi Harsaru Junction,Manesar Golf Course,AapnoGhar,Vega Schools NH-8,DLF Corporate Greens,Miracles Apollo Cradle Hospital,Hyatt Regency Gurugram,NH 48,Golden Greens Golf & Resorts Limited,Mount Olympus Junior School,Miracles Apollo Hospital,NH -8,"Savoy Suites, Manesar",Golden Greens Golf & Resorts,IMT Manesar,Amity University Gurugram,Golf Course Extension Road,"Dwarka Expy, Sector 109","Euro International School, Sector- 109",Jai Sai Ram Hospital,Aryan Hospital,Idea Cosmic Plaza,Indira Gandhi Intl Airport,Royal Institute Of Science & Management,Pataudi Road,Holiday Inn Sector 90,RPS International School,Aarvy Healthcare Hospital,Iris Broadway Mall,Imperia Mindspace,AIPL Business Tower,Heritage School,"Lotus Valley Intl School, Gurgaon",Gurugram University,Sector 55-56 Metro Station,Omaxe Gurgaon Mall,Sushant University,"Badshahpur Sohna Rd Hwy,Sector 48",Naurangpur Cricket Stadium,Naurangpur Road,National Highway 48,Vatika Town Square-INXT,Ompee International School,Manesar Bus Stand,Yashlok Medical Centre,Euro International School,WorldMark Gurgaon,Capital Cyberscape,The Shriram Millennium School,DoubleTree by Hilton Hotel,Badshahpur Sohna Hwy,Nakhrola Stadium,Delhi - Jaipur Expressway,Vatika Town Square-INXT Mall,Savoy Suites,Bal Bharati Public School,Vatika Business Centre,Indira Gandhi Int. Airport,St. Xavier's High School,Miracles Apollo Cradle,Ambience Mall New,NH8,Hyatt Regency Gurgaon,Delhi Public School,Elan Miracle Mall,Miracles Apollo Cradle Spectra Hospital,Agri Business Management Collage,Delhi Jaipur Expressway,Grand Hyatt Gurgaon,Duke Horse Riding Club,PVR Drive In Cinema,W Pratiksha Hospital,Metro World Mall,Unicosmos School,Faridabad Gurgaon Road,Sohna Road,Bestech Business Tower,Appu Ghar,SkyJumper Trampoline Park,Axis Bank,KMP Expressway,Karma Lakelands,Jungle Safari & Trails,DPS Manesar,Medanta Hospital,Faridabad - Gurgaon Road,Lingaya's Lalita Devi Institute,ASF Insignia SEZ,Banjara Market Gurugram,Central Plaza Mall,"Paras Hospitals, Gurgaon",Badshahpur Sohna Rd Hwy,Vega School,Indian School of Hospitality,Vatika City Centre,Aatish Hospital,Info Technology Park Phase 2,Huda Metro Station,Southern Peripheral Rd,Global Ways School,Radisson Hotel,NH 248A,Sector 55/56 Metro Station,Mavens Inn,Sanar International Hospital,Sector 53-54 Metro Station,"IILM University, Gurugram",The Banyan Tree World School,The Big Tree Cafe,DLF Golf and Country Club,"Delhi Public School, Sector 84",Aarvy Hospital,DPG Degree College,Shivani public school,Baghera University,Kutumbh Hospital,Bijwasan Railway Station,Global Foyer Mall,Phase 2 Metro Station,Gurgaon Dreamz Mall,"Metro Hospital, Palam Vihar",Delhi Ajmer Expressway,Infinity Business Park,Huda metro station,Rion's Hospital,"Euro International School, Sector- 109.",Golf Course Ext Road,"Heritage Xperiential Learning, CRPF Rd",Sector 54 Chowk Metro Station

In [38]:
location_df.columns[10:50]

Index(['Indira Gandhi International Airport', 'Hamoni Golf Camp',
       'Fun N Food Waterpark', 'Accenture DDC5', 'DPSG Palam Vihar Gurugram',
       'Park Hospital, Palam Vihar', 'Palam Vihar Halt Railway Station',
       'Dwarka Sector 21 Metro Station', 'Dwarka Expressway',
       'Fun N Food Water Park', 'Tau DeviLal Sports Complex', 'Hyatt Place',
       'Altrade Business Centre', 'AIPL Business Club Sector 62',
       'Heritage Xperiential Learning School', 'CK Birla Hospital',
       'Paras Trinity Mall Sector 63', 'Rapid Metro Station Sector 56',
       'De Adventure Park', 'Golf Course Ext Rd',
       'DoubleTree by Hilton Hotel Gurgaon',
       'KIIT College of Engineering Sohna Road', 'Mehrauli-Gurgaon Road',
       'Nirvana Rd', 'TERI Golf Course', 'The Shikshiyan School', 'WTC Plaza',
       'Luxus Haritma Resort', 'BSF Golf Course', 'Rions Hospital', 'Gurgaon',
       'Dwarka Sector 21', 'Nehru Stadium', 'Fun N Food WaterPark',
       'IGI Airport', 'Vasant Kunj', 'Prana

In [39]:
location_df.index = df.PropertyName

In [40]:
location_df.head()

,Bajghera Road,Palam Vihar Halt,DPSG Palam Vihar,Park Hospital,Gurgaon Railway Station,The NorthCap University,Dwarka Expy,Hyatt Place Gurgaon Udyog Vihar,"Dwarka Sector 21, Metro Station",Pacific D21 Mall,Indira Gandhi International Airport,Hamoni Golf Camp,Fun N Food Waterpark,Accenture DDC5,DPSG Palam Vihar Gurugram,"Park Hospital, Palam Vihar",Palam Vihar Halt Railway Station,Dwarka Sector 21 Metro Station,Dwarka Expressway,Fun N Food Water Park,Tau DeviLal Sports Complex,Hyatt Place,Altrade Business Centre,AIPL Business Club Sector 62,Heritage Xperiential Learning School,CK Birla Hospital,Paras Trinity Mall Sector 63,Rapid Metro Station Sector 56,De Adventure Park,Golf Course Ext Rd,DoubleTree by Hilton Hotel Gurgaon,KIIT College of Engineering Sohna Road,Mehrauli-Gurgaon Road,Nirvana Rd,TERI Golf Course,The Shikshiyan School,WTC Plaza,Luxus Haritma Resort,BSF Golf Course,Rions Hospital,Gurgaon,Dwarka Sector 21,Nehru Stadium,Fun N Food WaterPark,IGI Airport,Vasant Kunj,Pranavananda Int. School,DLF Site central office,Holiday Inn Gurugram Sector 90,Krishna Hospital,Royal Institute Of Science,Sapphire 83 Mall,NH48,Garhi Harsaru Junction,Manesar Golf Course,AapnoGhar,Vega Schools NH-8,DLF Corporate Greens,Miracles Apollo Cradle Hospital,Hyatt Regency Gurugram,NH 48,Golden Greens Golf & Resorts Limited,Mount Olympus Junior School,Miracles Apollo Hospital,NH -8,"Savoy Suites, Manesar",Golden Greens Golf & Resorts,IMT Manesar,Amity University Gurugram,Golf Course Extension Road,"Dwarka Expy, Sector 109","Euro International School, Sector- 109",Jai Sai Ram Hospital,Aryan Hospital,Idea Cosmic Plaza,Indira Gandhi Intl Airport,Royal Institute Of Science & Management,Pataudi Road,Holiday Inn Sector 90,RPS International School,Aarvy Healthcare Hospital,Iris Broadway Mall,Imperia Mindspace,AIPL Business Tower,Heritage School,"Lotus Valley Intl School, Gurgaon",Gurugram University,Sector 55-56 Metro Station,Omaxe Gurgaon Mall,Sushant University,"Badshahpur Sohna Rd Hwy,Sector 48",Naurangpur Cricket Stadium,Naurangpur Road,National Highway 48,Vatika Town Square-INXT,Ompee International School,Manesar Bus Stand,Yashlok Medical Centre,Euro International School,WorldMark Gurgaon,Capital Cyberscape,The Shriram Millennium School,DoubleTree by Hilton Hotel,Badshahpur Sohna Hwy,Nakhrola Stadium,Delhi - Jaipur Expressway,Vatika Town Square-INXT Mall,Savoy Suites,Bal Bharati Public School,Vatika Business Centre,Indira Gandhi Int. Airport,St. Xavier's High School,Miracles Apollo Cradle,Ambience Mall New,NH8,Hyatt Regency Gurgaon,Delhi Public School,Elan Miracle Mall,Miracles Apollo Cradle Spectra Hospital,Agri Business Management Collage,Delhi Jaipur Expressway,Grand Hyatt Gurgaon,Duke Horse Riding Club,PVR Drive In Cinema,W Pratiksha Hospital,Metro World Mall,Unicosmos School,Faridabad Gurgaon Road,Sohna Road,Bestech Business Tower,Appu Ghar,SkyJumper Trampoline Park,Axis Bank,KMP Expressway,Karma Lakelands,Jungle Safari & Trails,DPS Manesar,Medanta Hospital,Faridabad - Gurgaon Road,Lingaya's Lalita Devi Institute,ASF Insignia SEZ,Banjara Market Gurugram,Central Plaza Mall,"Paras Hospitals, Gurgaon",Badshahpur Sohna Rd Hwy,Vega School,Indian School of Hospitality,Vatika City Centre,Aatish Hospital,Info Technology Park Phase 2,Huda Metro Station,Southern Peripheral Rd,Global Ways School,Radisson Hotel,NH 248A,Sector 55/56 Metro Station,Mavens Inn,Sanar International Hospital,Sector 53-54 Metro Station,"IILM University, Gurugram",The Banyan Tree World School,The Big Tree Cafe,DLF Golf and Country Club,"Delhi Public School, Sector 84",Aarvy Hospital,DPG Degree College,Shivani public school,Baghera University,Kutumbh Hospital,Bijwasan Railway Station,Global Foyer Mall,Phase 2 Metro Station,Gurgaon Dreamz Mall,"Metro Hospital, Palam Vihar",Delhi Ajmer Expressway,Infinity Business Park,Huda metro station,Rion's Hospital,"Euro International School, Sector- 109.",Golf Course Ext Road,"Heritage Xperiential Learning, CRPF Rd",Sector 54 Chowk Metro Station

In [41]:
location_df.fillna(54000,inplace=True)

In [42]:
location_df.to_csv('location_df.csv')

In [43]:
location_df = pd.read_csv('../data/location_df_cleaned_raw.csv')

In [44]:
location_df.set_index('PropertyName', inplace=True)

In [45]:
location_df

Bajghera Road  Palam Vihar Halt  \
PropertyName                                                   
Smartworld One DXP                   800.0            2500.0   
M3M Crown                            550.0           54000.0   
Adani Brahma Samsara Vilasa         5300.0           54000.0   
Sobha City                          1500.0           54000.0   
Signature Global City 93           54000.0           54000.0   
...                                    ...               ...   
DLF Princeton Estate               54000.0           54000.0   
Pyramid Urban Homes 2              54000.0           54000.0   
Satya The Hermitage                54000.0           54000.0   
BPTP Spacio                        54000.0           54000.0   
SS The Coralwood                   54000.0           54000.0   

                             DPSG Palam Vihar  Park Hospital  \
PropertyName                                                   
Smartworld One DXP                     3100.0         3100.0   
M3M Crown                             54000.0        54000.0   
Adani Brahma Samsara Vilasa           54000.0        54000.0   
Sobha City                            54000.0        54000.0   
Signature Global City 93              54000.0         5500.0   
...                                       ...            ...   
DLF Princeton Estate                  54000.0        54000.0   
Pyramid Urban Homes 2                 54000.0        54000.0   
Satya The Hermitage                   54000.0        54000.0   
BPTP Spacio                           54000.0        54000.0   
SS The Coralwood                      54000.0        54000.0   

                             Gurgaon Railway Station  The NorthCap University  \
PropertyName                                                                    
Smartworld One DXP                            4900.0                   5400.0   
M3M Crown                                    54000.0                   6700.0   
Adani Brahma Samsara Vilasa                   2500.0                   8800.0   
Sobha City                                    6500.0                   6700.0   
Signature Global City 93                     54000.0                  54000.0   
...                                              ...                      ...   
DLF Princeton Estate                         54000.0                  54000.0   
Pyramid Urban Homes 2                        54000.0                  54000.0   
Satya The Hermitage                          54000.0                  54000.0   
BPTP Spacio                                  54000.0                  54000.0   
SS The Coralwood                             54000.0                  54000.0   

                             Dwarka Expressway  \
PropertyName                                     
Smartworld One DXP                      1200.0   
M3M Crown                               3800.0   
Adani Brahma Samsara Vilasa              700.0   
Sobha City                              5100.0   
Signature Global City 93               54000.0   
...                                        ...   
DLF Princeton Estate                   54000.0   
Pyramid Urban Homes 2                  54000.0   
Satya The Hermitage                    54000.0   
BPTP Spacio                            54000.0   
SS The Coralwood                       54000.0   

                             Hyatt Place Gurgaon Udyog Vihar  \
PropertyName                                                   
Smartworld One DXP                                    7700.0   
M3M Crown                                            54000.0   
Adani Brahma Samsara Vilasa                          54000.0   
Sobha City                                           54000.0   
Signature Global City 93                             54000.0   
...                                                      ...   
DLF Princeton Estate                                 54000.0   
Pyramid Urban Homes 2                                54000.0   
Satya The Hermitag

In [48]:
from sklearn.preprocessing import StandardScaler
# Initialize the scaler
scaler = StandardScaler()

# Apply the scaler to the entire dataframe
location_df_normalized = pd.DataFrame(scaler.fit_transform(location_df), columns=location_df.columns, index=location_df.index)

In [49]:
location_df_normalized

Bajghera Road  Palam Vihar Halt  \
PropertyName                                                   
Smartworld One DXP               -7.960979        -15.652476   
M3M Crown                        -7.998993          0.063888   
Adani Brahma Samsara Vilasa      -7.276720          0.063888   
Sobha City                       -7.854539          0.063888   
Signature Global City 93          0.128476          0.063888   
...                                    ...               ...   
DLF Princeton Estate              0.128476          0.063888   
Pyramid Urban Homes 2             0.128476          0.063888   
Satya The Hermitage               0.128476          0.063888   
BPTP Spacio                       0.128476          0.063888   
SS The Coralwood                  0.128476          0.063888   

                             DPSG Palam Vihar  Park Hospital  \
PropertyName                                                   
Smartworld One DXP                 -15.652476      -3.149592   
M3M Crown                            0.063888       0.328277   
Adani Brahma Samsara Vilasa          0.063888       0.328277   
Sobha City                           0.063888       0.328277   
Signature Global City 93             0.063888      -2.985606   
...                                       ...            ...   
DLF Princeton Estate                 0.063888       0.328277   
Pyramid Urban Homes 2                0.063888       0.328277   
Satya The Hermitage                  0.063888       0.328277   
BPTP Spacio                          0.063888       0.328277   
SS The Coralwood                     0.063888       0.328277   

                             Gurgaon Railway Station  The NorthCap University  \
PropertyName                                                                    
Smartworld One DXP                         -2.905447                -3.147217   
M3M Crown                                   0.375997                -3.054053   
Adani Brahma Samsara Vilasa                -3.065844                -2.903557   
Sobha City                                 -2.798516                -3.054053   
Signature Global City 93                    0.375997                 0.335688   
...                                              ...                      ...   
DLF Princeton Estate                        0.375997                 0.335688   
Pyramid Urban Homes 2                       0.375997                 0.335688   
Satya The Hermitage                         0.375997                 0.335688   
BPTP Spacio                                 0.375997                 0.335688   
SS The Coralwood                            0.375997                 0.335688   

                             Dwarka Expressway  \
PropertyName                                     
Smartworld One DXP                   -1.890423   
M3M Crown                            -1.769699   
Adani Brahma Samsara Vilasa          -1.913640   
Sobha City                           -1.709337   
Signature Global City 93              0.561210   
...                                        ...   
DLF Princeton Estate                  0.561210   
Pyramid Urban Homes 2                 0.561210   
Satya The Hermitage                   0.561210   
BPTP Spacio                           0.561210   
SS The Coralwood                      0.561210   

                             Hyatt Place Gurgaon Udyog Vihar  \
PropertyName                                                   
Smartworld One DXP                                -10.231739   
M3M Crown                                           0.090308   
Adani Brahma Samsara Vilasa                         0.090308   
Sobha City                                          0.090308   
Signature Global City 93                            0.090308   
...                                                      ...   
DLF Princeton Estate                                0.090308   
Pyramid Urban Homes 2                               0.090308   
Satya The Hermitag

In [50]:
cosine_sim3 = cosine_similarity(location_df_normalized)

In [51]:
cosine_sim3.shape

(246, 246)

In [57]:
def recommend_properties_with_scores(property_name, top_n=247):
    
    cosine_sim_matrix = 0.5*cosine_sim1 + 0.8*cosine_sim2 + 1*cosine_sim3
    # cosine_sim_matrix = cosine_sim3
    
    # Get the similarity scores for the property using its name as the index
    sim_scores = list(enumerate(cosine_sim_matrix[location_df_normalized.index.get_loc(property_name)]))
    
    # Sort properties based on the similarity scores
    sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get the indices and scores of the top_n most similar properties
    top_indices = [i[0] for i in sorted_scores[1:top_n+1]]
    top_scores = [i[1] for i in sorted_scores[1:top_n+1]]
    
    # Retrieve the names of the top properties using the indices
    top_properties = location_df_normalized.index[top_indices].tolist()
    
    # Create a dataframe with the results
    recommendations_df = pd.DataFrame({
        'PropertyName': top_properties,
        'SimilarityScore': top_scores
    })
    
    return recommendations_df

# Test the recommender function using a property name
recommend_properties_with_scores('Godrej Aria')

,PropertyName,SimilarityScore
0,M3M Sierra 68,0.876958
1,BPTP Spacio,0.836801
2,ILD Grand,0.831934
3,SS The Coralwood,0.766552
4,Unitech Fresco,0.746331
...,...,...
240,Elan The Presidential,-0.384885
241,Pioneer Urban Presidia,-0.399343
242,Emaar MGF Marbella,-0.416441
243,Tata Primanti,-0.425275


In [54]:
import pickle

In [83]:
pickle.dump(location_df, open('location_distance.pkl', 'wb'))

In [84]:
location_df[location_df['Bajghera Road'] < 2000]

,Bajghera Road,Palam Vihar Halt,DPSG Palam Vihar,Park Hospital,Gurgaon Railway Station,The NorthCap University,Dwarka Expressway,Hyatt Place Gurgaon Udyog Vihar,"Dwarka Sector 21, Metro Station",Pacific D21 Mall,Indira Gandhi International Airport,Hamoni Golf Camp,Fun N Food Water Park,Accenture DDC5,DPSG Palam Vihar Gurugram,"Park Hospital, Palam Vihar",Palam Vihar Halt Railway Station,Tau DeviLal Sports Complex,Hyatt Place,Altrade Business Centre,AIPL Business Club Sector 62,Heritage Xperiential Learning School,CK Birla Hospital,Paras Trinity Mall Sector 63,Rapid Metro Station Sector 56,De Adventure Park,Golf Course Ext Road,DoubleTree by Hilton Hotel Gurgaon,KIIT College of Engineering Sohna Road,Mehrauli-Gurgaon Road,Nirvana Rd,TERI Golf Course,The Shikshiyan School,WTC Plaza,Luxus Haritma Resort,BSF Golf Course,Rions Hospital,Gurgaon,Dwarka Sector 21,Nehru Stadium,IGI Airport,Vasant Kunj,Pranavananda International School,DLF Site central office,Holiday Inn Gurugram Sector 90,Krishna Hospital,Royal Institute Of Science,Sapphire 83 Mall,NH 48,Garhi Harsaru Junction,Manesar Golf Course,AapnoGhar,Vega Schools NH-8,DLF Corporate Greens,Miracles Apollo Cradle Hospital,Hyatt Regency Gurugram,Golden Greens Golf & Resorts Limited,Mount Olympus Junior School,Miracles Apollo Hospital,NH -8,"Savoy Suites, Manesar",Golden Greens Golf & Resorts,IMT Manesar,Amity University Gurugram,Golf Course Extension Road,"Dwarka Expy, Sector 109","Euro International School, Sector- 109.",Jai Sai Ram Hospital,Aryan Hospital,Idea Cosmic Plaza,Royal Institute Of Science & Management,Pataudi Road,Holiday Inn Sector 90,RPS International School,Aarvy Healthcare Hospital,Iris Broadway Mall,Imperia Mindspace,AIPL Business Tower,Heritage School,"Lotus Valley Intl School, Gurgaon",Gurugram University,Sector 55-56 Metro Station,Omaxe Gurgaon Mall,Sushant University,"Badshahpur Sohna Rd Hwy,Sector 48",Naurangpur Cricket Stadium,Naurangpur Road,National Highway 48,Vatika Town Square-INXT,Ompee International School,Manesar Bus Stand,Yashlok Medical Centre,Euro International School,WorldMark Gurgaon,Capital Cyberscape,The Shriram Millennium School,DoubleTree by Hilton Hotel,Badshahpur Sohna Hwy,Nakhrola Stadium,Delhi - Jaipur Expressway,Vatika Town Square-INXT Mall,Savoy Suites,Bal Bharati Public School,Vatika Business Centre,St. Xavier's High School,Miracles Apollo Cradle,Ambience Mall New,Hyatt Regency Gurgaon,Delhi Public School,Elan Miracle Mall,Miracles Apollo Cradle Spectra Hospital,Agri Business Management Collage,Grand Hyatt Gurgaon,Duke Horse Riding Club,PVR Drive In Cinema,W Pratiksha Hospital,Metro World Mall,Unicosmos School,Faridabad - Gurgaon Road,Sohna Road,Bestech Business Tower,Appu Ghar,SkyJumper Trampoline Park,Axis Bank,KMP Expressway,Karma Lakelands,Jungle Safari & Trails,DPS Manesar,Medanta Hospital,Lingaya's Lalita Devi Institute,ASF Insignia SEZ,Banjara Market Gurugram,Central Plaza Mall,"Paras Hospitals, Gurgaon",Badshahpur Sohna Rd Hwy,Vega School,Indian School of Hospitality,Vatika City Centre,Aatish Hospital,Info Technology Park Phase 2,Huda Metro Station,Southern Peripheral Road,Global Ways School,Radisson Hotel,NH 248A,Mavens Inn,Sanar International Hospital,Sector 53-54 Metro Station,"IILM University, Gurugram",The Banyan Tree World School,The Big Tree Cafe,DLF Golf and Country Club,"Delhi Public School, Sector 84",Aarvy Hospital,DPG Degree College,Shivani public school,Baghera University,Kutumbh Hospital,Bijwasan Railway Station,Global Foyer Mall,Phase 2 Metro Station,Gurgaon Dreamz Mall,"Metro Hospital, Palam Vihar",Delhi Ajmer Expressway,Infinity Business Park,Rion's Hospital,"Heritage Xperiential Learning, CRPF Rd",Sector 54 Chowk Metro Station,Gurgaon - Delhi Expy,Genesis Hospital Sector 84,DPGITM Engineering College Sector 34,Sapphire 83 Mall Sector 83,Holiday Inn Hotel Sector 90,SkyJumper Trampoline Park Gurgaon,National Tennis Academy Sector 98,NH-8 Delhi Jaipur Highway,Nouveau Medics Multispeciality OPD,Heritage Village Res

In [85]:
location_df.columns.tolist()

['Bajghera Road',
 'Palam Vihar Halt',
 'DPSG Palam Vihar',
 'Park Hospital',
 'Gurgaon Railway Station',
 'The NorthCap University',
 'Dwarka Expressway',
 'Hyatt Place Gurgaon Udyog Vihar',
 'Dwarka Sector 21, Metro Station',
 'Pacific D21 Mall',
 'Indira Gandhi International Airport',
 'Hamoni Golf Camp',
 'Fun N Food Water Park',
 'Accenture DDC5',
 'DPSG Palam Vihar Gurugram',
 'Park Hospital, Palam Vihar',
 'Palam Vihar Halt Railway Station',
 'Tau DeviLal Sports Complex',
 'Hyatt Place',
 'Altrade Business Centre',
 'AIPL Business Club Sector 62',
 'Heritage Xperiential Learning School',
 'CK Birla Hospital',
 'Paras Trinity Mall Sector 63',
 'Rapid Metro Station Sector 56',
 'De Adventure Park',
 'Golf Course Ext Road',
 'DoubleTree by Hilton Hotel Gurgaon',
 'KIIT College of Engineering Sohna Road',
 'Mehrauli-Gurgaon Road',
 'Nirvana Rd',
 'TERI Golf Course',
 'The Shikshiyan School',
 'WTC Plaza',
 'Luxus Haritma Resort',
 'BSF Golf Course',
 'Rions Hospital',
 'Gurgaon',
 '

In [55]:
cosine_sim_matrix = 0.5*cosine_sim1 + 0.8*cosine_sim2 + 1*cosine_sim3

In [56]:
import pickle
pickle.dump(cosine_sim_matrix, open('../data/cosine_sim_matrix.pkl', 'wb'))
